# Step 1: Install Dependencies
This cell installs the necessary libraries for our RAG system: `transformers` for the LLM, `sentence-transformers` for embeddings, `faiss-cpu` for vector search, and `datasets` to load our data.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
!pip install -q transformers sentence-transformers faiss-cpu datasets

# Step 2: Load the Dataset
We are loading a demo text dataset and preparing the documents by combining the prompts and completions.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ka1d0/demo-text-dataset")
data = dataset["train"]

documents = [row["prompt"] + " " + row["completion"] for row in data]

print(f"Loaded {len(documents)} documents.")
print("Sample:", documents[:1])

Loaded 20 documents.
Sample: ['Name: Alice, Purchase: Laptop, Category: Electronics, Total Products: 5 Hi Alice, you recently purchased a Laptop from the Electronics category. Enjoy 10% off on similar electronics now!']


# Step 3: Create Document Embeddings
We use a SentenceTransformer model (`all-MiniLM-L6-v2`) to convert our text documents into numerical vectors.

In [ ]:
from sentence_transformers import SentenceTransformer

# Initialize the embedder
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings for our documents
doc_embeddings = embedder.encode(documents)
print(f"Embeddings shape: {doc_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (20, 384)


# Step 4: Build the Vector Index
We use FAISS (Facebook AI Similarity Search) to create a searchable index of our document vectors for fast retrieval.

In [ ]:
import faiss
import numpy as np

# Dimension of the embeddings
dimension = doc_embeddings.shape[1]

# Create the FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

# Step 5: Define the Retrieval Function
This function takes a query, converts it to an embedding, and finds the most relevant documents from the FAISS index.

In [ ]:
def retrieve(query, top_k=3):
    # Encode query using the same 'embedder' instance
    query_embedding = embedder.encode([query])

    # Search in FAISS index
    distances, indices = index.search(np.array(query_embedding), top_k)

    # Map indices back to document text
    results = [documents[i] for i in indices[0]]

    return results

# Step 6: Load the Language Model
We initialize the TinyLlama model, which will act as our generator to answer questions based on the retrieved context.

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# Step 7: Define the Generation Logic
This function combines retrieval and generation. It fetches context and uses a strict prompt to ensure the LLM answers accurately.

In [ ]:
def generate_answer(query):
    retrieved_docs = retrieve(query)

    context = " ".join(retrieved_docs)

    prompt = f"""
    You are a strict question answering system.

    Rules:
    - Answer ONLY the question
    - Return ONLY the exact answer
    - Do NOT list multiple items
    - Do NOT repeat context
    - Output should be one short phrase

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = generator(
        prompt,
        max_new_tokens=20,
        temperature=0.0,
        do_sample=False,
        truncation=True
    )

    text = response[0]["generated_text"]
    answer = text.split("Answer:")[-1].strip()

    return answer, retrieved_docs

# Step 8: Testing and Fallback Extraction
This cell contains a test query and a fallback function to ensure ground-truth accuracy by extracting data directly from the text if needed.

In [ ]:
import re

def extract_answer(docs, query):
    query = query.lower()

    for doc in docs:
        # check for common names in query and matching document
        for name in ["Alice", "Bob", "Carol", "David", "Emma", "Frank", "Grace", "Hank", "Ivy", "Jack", "Kate", "Leo", "Mona", "Nate", "Olive", "Paul", "Quinn", "Rose", "Sam", "Tina"]:
            if name.lower() in query and name in doc:
                match = re.search(r'Purchase:\s*(.*?),', doc)
                if match:
                    return match.group(1)
    return None

query = "What did Alice purchase?"

answer, docs = generate_answer(query)

# Optional extraction fallback
fallback = extract_answer(docs, query)
if fallback:
    answer = fallback

print("Question:", query)
print("\nRetrieved Docs:", docs)
print("\nFinal Answer:", answer)

Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What did Alice purchase?

Retrieved Docs: ['Name: Alice, Purchase: Laptop, Category: Electronics, Total Products: 5 Hi Alice, you recently purchased a Laptop from the Electronics category. Enjoy 10% off on similar electronics now!', 'Name: Emma, Purchase: T-shirt, Category: Clothing, Total Products: 4 Hi Emma, you bought a T-shirt from Clothing. Enjoy 10% off on more clothing items!', 'Name: Tina, Purchase: Earrings, Category: Accessories, Total Products: 1 Hi Tina, you recently bought Earrings from Accessories. Get 10% off on more accessories today!']

Final Answer: Laptop


# Step 9: Interactive RAG Chatbot
Run this cell to start a continuous chat loop where you can ask multiple questions about the dataset.

In [ ]:
print("🤖 RAG Chatbot is running (type 'exit' to stop)\n")

while True:
    try:
        query = input("You: ").strip()

        if query.lower() in ["exit", "quit"]:
            print("Bot: Goodbye 👋")
            break

        if query == "":
            continue

        answer, docs = generate_answer(query)

        # Use the smarter fallback
        fallback = extract_answer(docs, query)
        if fallback:
            answer = fallback

        print("\n🔎 Top Retrieved Docs:")
        for i, doc in enumerate(docs):
            print(f"{i+1}. {doc[:80]}...")

        print(f"\n🤖 Answer: {answer}")
        print("\n" + "="*60 + "\n")

    except KeyboardInterrupt:
        print("\nBot: Stopped manually 👋")
        break

🤖 RAG Chatbot is running (type 'exit' to stop)

You: What did Emma buy?


Both `max_new_tokens` (=20) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔎 Top Retrieved Docs:
1. Name: Emma, Purchase: T-shirt, Category: Clothing, Total Products: 4 Hi Emma, yo...
2. Name: Kate, Purchase: Winter Coat, Category: Clothing, Total Products: 3 Hi Kate...
3. Name: Tina, Purchase: Earrings, Category: Accessories, Total Products: 1 Hi Tina...

🤖 Answer: T-shirt



Bot: Stopped manually 👋
